In [ ]:
import torch
import math


In [2]:
import torch

# ==========================================
# 1. DATA AND VOCABULARY
# ==========================================
raw_data = [
    ("I like deep learning", "mujhe deep learning pasand hai"),
    ("I eat biryani", "main biryani khata hoon"),
    ("he plays cricket", "wo cricket khelta hai"),
    ("I play hockey", "main hockey khelta hoon"),
    ("I read research papers", "main research papers parhta hoon"),
    ("I want a scholarship", "mujhe scholarship chahiye"),
    ("I live in Pakistan", "main pakistan mein rehta hoon"),
    ("roses are beautiful", "gulab khubsurat hain"),
    ("I am taking the academic test", "main academic test de raha hoon")
]

def build_vocab(sentences):
    vocab = {"<SOS>": 0, "<EOS>": 1}
    for sentence in sentences:
        for word in sentence.lower().split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

english_sentences = [pair[0] for pair in raw_data]
urdu_sentences = [pair[1] for pair in raw_data]

eng_vocab = build_vocab(english_sentences)
urdu_vocab = build_vocab(urdu_sentences)
urdu_idx_to_word = {idx: word for word, idx in urdu_vocab.items()}

# ==========================================
# 2. NEURAL NETWORK COMPONENTS
# ==========================================

class Embedding:
    def __init__(self, vocab_size, embed_dim):
        self.E = (torch.randn(vocab_size, embed_dim) * 0.1).requires_grad_()

    def forward(self, idx):
        return self.E[idx]
        
    def update_weights(self, lr):
        with torch.no_grad():
            if self.E.grad is not None:
                self.E -= lr * self.E.grad
                self.E.grad.zero_()

class LSTM:
    def __init__(self, input_dim, hidden_dim):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        concat_dim = input_dim + hidden_dim

        self.W_f, self.b_f = self.gate(concat_dim, hidden_dim, forget_bias=True) 
        self.W_i, self.b_i = self.gate(concat_dim, hidden_dim) 
        self.W_c, self.b_c = self.gate(concat_dim, hidden_dim) 
        self.W_o, self.b_o = self.gate(concat_dim, hidden_dim) 

    def gate(self, input_size, output_size, forget_bias=False):
        W = (torch.randn(input_size, output_size) * 0.1).requires_grad_()
        b = (torch.randn(1, output_size) * 0.1).requires_grad_()
        if forget_bias:
            with torch.no_grad():
                b += 1.0 # Prevent vanishing memory
        return W, b

    def forward(self, X, prev_c, prev_h):
        xh = torch.cat([X, prev_h], dim=1)
        
        f = torch.sigmoid(xh @ self.W_f + self.b_f)
        i = torch.sigmoid(xh @ self.W_i + self.b_i)
        c_tilde = torch.tanh(xh @ self.W_c + self.b_c)
        o = torch.sigmoid(xh @ self.W_o + self.b_o)
        
        c_next = f * prev_c + i * c_tilde
        h_next = o * torch.tanh(c_next)
        return c_next, h_next

    def update_weights(self, lr):
        with torch.no_grad():
            for param in [self.W_f, self.b_f, self.W_i, self.b_i, self.W_c, self.b_c, self.W_o, self.b_o]:
                if param.grad is not None:
                    param -= lr * param.grad
                    param.grad.zero_()

class BahdanauAttention:
    def __init__(self, hidden_dim):
        self.W_q = (torch.randn(hidden_dim, hidden_dim) * 0.1).requires_grad_()
        self.W_k = (torch.randn(hidden_dim, hidden_dim) * 0.1).requires_grad_()
        self.V = (torch.randn(hidden_dim, 1) * 0.1).requires_grad_()

    def forward(self, query, keys):
        # query: (1, hidden_dim) | keys: (src_seq_len, hidden_dim)
        alignment_features = torch.tanh(query @ self.W_q + keys @ self.W_k)
        scores = alignment_features @ self.V
        scores = scores.T # (1, src_seq_len)
        
        # Softmax
        exp_scores = torch.exp(scores)
        attn_weights = exp_scores / torch.sum(exp_scores, dim=1, keepdim=True)
        
        # Context Vector
        context_vector = attn_weights @ keys
        return context_vector, attn_weights

    def update_weights(self, lr):
        with torch.no_grad():
            for param in [self.W_q, self.W_k, self.V]:
                if param.grad is not None:
                    param -= lr * param.grad
                    param.grad.zero_()

class ANN:
    def __init__(self, hidden_dim, output_vocab_size):
        self.W_y = (torch.randn(hidden_dim, output_vocab_size) * 0.1).requires_grad_()
        self.b_y = (torch.randn(1, output_vocab_size) * 0.1).requires_grad_()

    def forward(self, h):
        logits = h @ self.W_y + self.b_y
        exp_logits = torch.exp(logits)
        probabilities = exp_logits / torch.sum(exp_logits, dim=1, keepdim=True)
        return probabilities
        
    def update_weights(self, lr):
        with torch.no_grad():
            for param in [self.W_y, self.b_y]:
                if param.grad is not None:
                    param -= lr * param.grad
                    param.grad.zero_()

class CrossEntropyLoss:
    def forward(self, predictions, targets):
        eps = 1e-9
        seq_len = targets.shape[0] 
        loss = -torch.sum(targets * torch.log(predictions + eps)) / seq_len
        return loss

# ==========================================
# 3. ORCHESTRATOR
# ==========================================

class Seq2seq:
    def __init__(self, eng_vocab_size, urdu_vocab_size, embed_dim, hidden_dim):
        self.hidden_dim = hidden_dim
        
        self.src_embed = Embedding(eng_vocab_size, embed_dim)
        self.tgt_embed = Embedding(urdu_vocab_size, embed_dim)
        
        self.encoder = LSTM(embed_dim, hidden_dim)
        # Decoder input is word embedding + context vector
        self.decoder = LSTM(embed_dim + hidden_dim, hidden_dim) 
        
        self.attention = BahdanauAttention(hidden_dim)
        self.ANN = ANN(hidden_dim, urdu_vocab_size)

    def forward(self, src_indices, tgt_indices):
        self.c_s = torch.zeros(1, self.hidden_dim)
        self.h_s = torch.zeros(1, self.hidden_dim)

        # Encoder pass
        encoder_outputs = []
        for idx in src_indices:
            x_t = self.src_embed.forward(idx).unsqueeze(0) 
            self.c_s, self.h_s = self.encoder.forward(x_t, self.c_s, self.h_s)
            encoder_outputs.append(self.h_s) 
            
        encoder_outputs = torch.cat(encoder_outputs, dim=0) 

        # Decoder pass (Teacher Forcing)
        outputs = [] 
        for idx in tgt_indices[:-1]:
            y_t = self.tgt_embed.forward(idx).unsqueeze(0) 
            
            # Bahdanau: Attention using previous hidden state
            context_vector, _ = self.attention.forward(self.h_s, encoder_outputs)
            
            # Concatenate embedding and context vector
            lstm_input = torch.cat([y_t, context_vector], dim=1)
            
            self.c_s, self.h_s = self.decoder.forward(lstm_input, self.c_s, self.h_s)
            y_hat = self.ANN.forward(self.h_s)
            outputs.append(y_hat)
            
        return torch.cat(outputs, dim=0)
        
    def update_weights(self, lr):
        self.src_embed.update_weights(lr)
        self.tgt_embed.update_weights(lr)
        self.encoder.update_weights(lr)
        self.decoder.update_weights(lr)
        self.attention.update_weights(lr)
        self.ANN.update_weights(lr)

# ==========================================
# 4. TRAINING AND INFERENCE
# ==========================================

embed_dim = 16
hidden_dim = 32

model = Seq2seq(len(eng_vocab), len(urdu_vocab), embed_dim, hidden_dim)
criterion = CrossEntropyLoss()

epochs = 300
learning_rate = 0.5

print("Starting Autograd Training...\n")

for epoch in range(epochs):
    epoch_loss = 0
    
    for eng_sent, urdu_sent in raw_data:
        src_indices = [eng_vocab[w] for w in eng_sent.lower().split()]
        tgt_indices = [urdu_vocab["<SOS>"]] + [urdu_vocab[w] for w in urdu_sent.lower().split()] + [urdu_vocab["<EOS>"]]
        
        # 1. Forward Pass
        predictions = model.forward(src_indices, tgt_indices)
        
        # 2. Compute targets and Loss
        target_words_to_predict = tgt_indices[1:]
        Y_one_hot = torch.zeros(len(target_words_to_predict), len(urdu_vocab))
        for i, word_idx in enumerate(target_words_to_predict):
            Y_one_hot[i, word_idx] = 1.0
            
        loss = criterion.forward(predictions, Y_one_hot)
        epoch_loss += loss.item()
        
        # 3. Autograd Backward Pass & Manual Update
        loss.backward()
        model.update_weights(lr=learning_rate)
        
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Average Loss: {(epoch_loss/len(raw_data)):.4f}")

# Inference Function
def translate(model, english_sentence, max_length=15):
    words = english_sentence.lower().split()
    src_indices = [eng_vocab[w] for w in words if w in eng_vocab]
    
    model.c_s = torch.zeros(1, model.hidden_dim)
    model.h_s = torch.zeros(1, model.hidden_dim)

    encoder_outputs = [] 
    for idx in src_indices:
        x_t = model.src_embed.forward(idx).unsqueeze(0) 
        model.c_s, model.h_s = model.encoder.forward(x_t, model.c_s, model.h_s)
        encoder_outputs.append(model.h_s) 
        
    encoder_outputs = torch.cat(encoder_outputs, dim=0) 

    predicted_words = []
    current_word_idx = urdu_vocab["<SOS>"]
    
    for t in range(max_length):
        y_t = model.tgt_embed.forward(current_word_idx).unsqueeze(0)
        
        context_vector, _ = model.attention.forward(model.h_s, encoder_outputs)
        lstm_input = torch.cat([y_t, context_vector], dim=1)
        
        model.c_s, model.h_s = model.decoder.forward(lstm_input, model.c_s, model.h_s)
        
        probabilities = model.ANN.forward(model.h_s)
        best_guess_idx = torch.argmax(probabilities, dim=1).item()
        
        if best_guess_idx == urdu_vocab["<EOS>"]:
            break
            
        predicted_words.append(urdu_idx_to_word[best_guess_idx])
        current_word_idx = best_guess_idx

    return " ".join(predicted_words)



Starting Autograd Training...

Epoch 50/300 | Average Loss: 0.5011
Epoch 100/300 | Average Loss: 0.0283
Epoch 150/300 | Average Loss: 0.0131
Epoch 200/300 | Average Loss: 0.0084
Epoch 250/300 | Average Loss: 0.0062
Epoch 300/300 | Average Loss: 0.0048

Testing the Translator:
English: I like deep learning
Urdu:    mujhe deep learning pasand hai

English: I live in Pakistan
Urdu:    main pakistan mein rehta hoon

English: he plays hockey
Urdu:    wo cricket khelta hai



In [3]:
print("\nTesting the Translator:")
test_sentences = [
    "I like deep learning",
    "I live in Pakistan",
    "Pakistan is beautiful"
]

for sentence in test_sentences:
    print(f"English: {sentence}")
    print(f"Urdu:    {translate(model, sentence)}\n")


Testing the Translator:
English: I like deep learning
Urdu:    mujhe deep learning pasand hai

English: I live in Pakistan
Urdu:    main pakistan mein rehta hoon

English: Pakistan is beautiful
Urdu:    gulab khubsurat hain



In [6]:
import torch

# ==========================================
# 1. DATA AND VOCABULARY (Unchanged)
# ==========================================
raw_data = [
    ("I like deep learning", "mujhe deep learning pasand hai"),
    ("I eat biryani", "main biryani khata hoon"),
    ("he plays cricket", "wo cricket khelta hai"),
    ("I play hockey", "main hockey khelta hoon"),
    ("I read research papers", "main research papers parhta hoon"),
    ("I want a scholarship", "mujhe scholarship chahiye"),
    ("I live in Pakistan", "main pakistan mein rehta hoon"),
    ("roses are beautiful", "gulab khubsurat hain"),
    ("I am taking the academic test", "main academic test de raha hoon")
]

def build_vocab(sentences):
    vocab = {"<SOS>": 0, "<EOS>": 1}
    for sentence in sentences:
        for word in sentence.lower().split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

english_sentences = [pair[0] for pair in raw_data]
urdu_sentences = [pair[1] for pair in raw_data]

eng_vocab = build_vocab(english_sentences)
urdu_vocab = build_vocab(urdu_sentences)
urdu_idx_to_word = {idx: word for word, idx in urdu_vocab.items()}

# ==========================================
# 2. NEURAL NETWORK COMPONENTS (Manual Autograd)
# ==========================================

class Embedding:
    def __init__(self, vocab_size, embed_dim):
        self.E = torch.randn(vocab_size, embed_dim) * 0.1
        self.grad_E = torch.zeros_like(self.E)

    def forward(self, idx):
        return self.E[idx]
        
    def backward(self, d_out, idx):
        # Accumulate gradient for the specific word index
        self.grad_E[idx] += d_out.squeeze(0)

    def update_weights(self, lr):
        self.E -= lr * self.grad_E
        self.grad_E.zero_()

class LSTM:
    def __init__(self, input_dim, hidden_dim):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        concat_dim = input_dim + hidden_dim

        # No requires_grad_()
        self.W_f, self.b_f = self.gate(concat_dim, hidden_dim, forget_bias=True) 
        self.W_i, self.b_i = self.gate(concat_dim, hidden_dim) 
        self.W_c, self.b_c = self.gate(concat_dim, hidden_dim) 
        self.W_o, self.b_o = self.gate(concat_dim, hidden_dim) 
        
        self.zero_grad()

    def zero_grad(self):
        self.grad_W_f = torch.zeros_like(self.W_f)
        self.grad_b_f = torch.zeros_like(self.b_f)
        self.grad_W_i = torch.zeros_like(self.W_i)
        self.grad_b_i = torch.zeros_like(self.b_i)
        self.grad_W_c = torch.zeros_like(self.W_c)
        self.grad_b_c = torch.zeros_like(self.b_c)
        self.grad_W_o = torch.zeros_like(self.W_o)
        self.grad_b_o = torch.zeros_like(self.b_o)

    def gate(self, input_size, output_size, forget_bias=False):
        W = torch.randn(input_size, output_size) * 0.1
        b = torch.randn(1, output_size) * 0.1
        if forget_bias:
            b += 1.0 
        return W, b

    def forward(self, X, prev_c, prev_h):
        xh = torch.cat([X, prev_h], dim=1)
        
        f = torch.sigmoid(xh @ self.W_f + self.b_f)
        i = torch.sigmoid(xh @ self.W_i + self.b_i)
        c_tilde = torch.tanh(xh @ self.W_c + self.b_c) # User notes called this 'k'
        o = torch.sigmoid(xh @ self.W_o + self.b_o)
        
        c_next = f * prev_c + i * c_tilde
        h_next = o * torch.tanh(c_next)
        
        # Cache for BPTT
        cache = (prev_c, prev_h, xh, f, i, c_tilde, o, c_next)
        return c_next, h_next, cache

    def backward(self, dh_next, dc_next, cache):
        prev_c, prev_h, xh, f, i, c_tilde, o, c_next = cache

        # Hidden state gradient splits into output gate and cell state
        do = dh_next * torch.tanh(c_next)
        dc_local = dh_next * o * (1 - torch.tanh(c_next)**2)
        
        # Total cell state gradient (local from h + from future time step)
        dc_total = dc_local + dc_next

        # Gate gradients (chain rule through cell state updates)
        df = dc_total * prev_c
        di = dc_total * c_tilde
        dc_tilde = dc_total * i
        dc_prev = dc_total * f

        # Activation derivatives
        dz_f = df * f * (1 - f)
        dz_i = di * i * (1 - i)
        dz_c = dc_tilde * (1 - c_tilde**2)
        dz_o = do * o * (1 - o)

        # Accumulate Weight Gradients
        self.grad_W_f += xh.T @ dz_f
        self.grad_b_f += torch.sum(dz_f, dim=0, keepdim=True)
        self.grad_W_i += xh.T @ dz_i
        self.grad_b_i += torch.sum(dz_i, dim=0, keepdim=True)
        self.grad_W_c += xh.T @ dz_c
        self.grad_b_c += torch.sum(dz_c, dim=0, keepdim=True)
        self.grad_W_o += xh.T @ dz_o
        self.grad_b_o += torch.sum(dz_o, dim=0, keepdim=True)

        # Gradient flowing back to inputs (X and prev_h)
        dxh = (dz_f @ self.W_f.T) + (dz_i @ self.W_i.T) + \
              (dz_c @ self.W_c.T) + (dz_o @ self.W_o.T)

        dX = dxh[:, :self.input_dim]
        dprev_h = dxh[:, self.input_dim:]

        return dX, dc_prev, dprev_h

    def update_weights(self, lr):
        self.W_f -= lr * self.grad_W_f
        self.b_f -= lr * self.grad_b_f
        self.W_i -= lr * self.grad_W_i
        self.b_i -= lr * self.grad_b_i
        self.W_c -= lr * self.grad_W_c
        self.b_c -= lr * self.grad_b_c
        self.W_o -= lr * self.grad_W_o
        self.b_o -= lr * self.grad_b_o
        self.zero_grad()

class BahdanauAttention:
    def __init__(self, hidden_dim):
        self.W_q = torch.randn(hidden_dim, hidden_dim) * 0.1
        self.W_k = torch.randn(hidden_dim, hidden_dim) * 0.1
        self.V = torch.randn(hidden_dim, 1) * 0.1
        self.zero_grad()

    def zero_grad(self):
        self.grad_W_q = torch.zeros_like(self.W_q)
        self.grad_W_k = torch.zeros_like(self.W_k)
        self.grad_V = torch.zeros_like(self.V)

    def forward(self, query, keys):
        q_proj = query @ self.W_q
        k_proj = keys @ self.W_k
        alignment_features = torch.tanh(q_proj + k_proj)
        
        scores = alignment_features @ self.V
        scores = scores.T # (1, src_seq_len)
        
        # Softmax
        exp_scores = torch.exp(scores)
        attn_weights = exp_scores / torch.sum(exp_scores, dim=1, keepdim=True)
        
        # Context Vector
        context_vector = attn_weights @ keys
        
        cache = (query, keys, alignment_features, scores, attn_weights)
        return context_vector, cache

    def backward(self, dContext, cache):
        query, keys, alignment_features, scores, attn_weights = cache
        
        # Context Vector backward: C = alpha @ keys
        d_attn_weights = dContext @ keys.T  # (1, seq_len)
        dKeys_from_C = attn_weights.T @ dContext # (seq_len, hidden_dim)

        # Softmax backward
        d_scores = attn_weights * (d_attn_weights - torch.sum(attn_weights * d_attn_weights, dim=1, keepdim=True))
        d_scores = d_scores.T # (seq_len, 1)

        # Output Layer backward: e = Z @ V
        self.grad_V += alignment_features.T @ d_scores
        d_alignment = d_scores @ self.V.T # (seq_len, hidden_dim)

        # Tanh backward
        d_pre_align = d_alignment * (1 - alignment_features**2)

        # Linear Projections backward
        self.grad_W_k += keys.T @ d_pre_align
        dKeys_from_align = d_pre_align @ self.W_k.T

        # Query uses broadcasting, so we must sum over the sequence length
        self.grad_W_q += query.T @ torch.sum(d_pre_align, dim=0, keepdim=True)
        dQuery = torch.sum(d_pre_align @ self.W_q.T, dim=0, keepdim=True)

        # Total gradient for keys (Encoder states)
        dKeys = dKeys_from_C + dKeys_from_align

        return dQuery, dKeys

    def update_weights(self, lr):
        self.W_q -= lr * self.grad_W_q
        self.W_k -= lr * self.grad_W_k
        self.V -= lr * self.grad_V
        self.zero_grad()

class ANN:
    def __init__(self, hidden_dim, output_vocab_size):
        self.W_y = torch.randn(hidden_dim, output_vocab_size) * 0.1
        self.b_y = torch.randn(1, output_vocab_size) * 0.1
        self.zero_grad()

    def zero_grad(self):
        self.grad_W_y = torch.zeros_like(self.W_y)
        self.grad_b_y = torch.zeros_like(self.b_y)

    def forward(self, h):
        logits = h @ self.W_y + self.b_y
        exp_logits = torch.exp(logits - torch.max(logits, dim=1, keepdim=True)[0]) # Stable softmax
        probabilities = exp_logits / torch.sum(exp_logits, dim=1, keepdim=True)
        cache = h
        return probabilities, cache
        
    def backward(self, dZ, cache):
        h = cache
        self.grad_W_y += h.T @ dZ
        self.grad_b_y += torch.sum(dZ, dim=0, keepdim=True)
        dh = dZ @ self.W_y.T
        return dh

    def update_weights(self, lr):
        self.W_y -= lr * self.grad_W_y
        self.b_y -= lr * self.grad_b_y
        self.zero_grad()

class CrossEntropyLoss:
    # We only use this to compute the scalar loss value for logging now.
    # The derivative (probs - targets) is handled directly in the Seq2seq backward loop!
    def forward(self, predictions, targets):
        eps = 1e-9
        seq_len = targets.shape[0] 
        loss = -torch.sum(targets * torch.log(predictions + eps)) / seq_len
        return loss

# ==========================================
# 3. ORCHESTRATOR
# ==========================================

class Seq2seq:
    def __init__(self, eng_vocab_size, urdu_vocab_size, embed_dim, hidden_dim):
        self.hidden_dim = hidden_dim
        
        self.src_embed = Embedding(eng_vocab_size, embed_dim)
        self.tgt_embed = Embedding(urdu_vocab_size, embed_dim)
        
        self.encoder = LSTM(embed_dim, hidden_dim)
        self.decoder = LSTM(embed_dim + hidden_dim, hidden_dim) 
        
        self.attention = BahdanauAttention(hidden_dim)
        self.ANN = ANN(hidden_dim, urdu_vocab_size)

    def forward(self, src_indices, tgt_indices):
        self.c_s = torch.zeros(1, self.hidden_dim)
        self.h_s = torch.zeros(1, self.hidden_dim)

        encoder_outputs = []
        encoder_caches = []
        
        for idx in src_indices:
            x_t = self.src_embed.forward(idx).unsqueeze(0) 
            self.c_s, self.h_s, enc_cache = self.encoder.forward(x_t, self.c_s, self.h_s)
            encoder_outputs.append(self.h_s) 
            encoder_caches.append((idx, enc_cache))
            
        encoder_outputs = torch.cat(encoder_outputs, dim=0) 

        outputs = [] 
        decoder_caches = []
        
        for i, idx in enumerate(tgt_indices[:-1]):
            y_t = self.tgt_embed.forward(idx).unsqueeze(0) 
            
            context_vector, attn_cache = self.attention.forward(self.h_s, encoder_outputs)
            lstm_input = torch.cat([y_t, context_vector], dim=1)
            
            self.c_s, self.h_s, dec_cache = self.decoder.forward(lstm_input, self.c_s, self.h_s)
            y_hat, ann_cache = self.ANN.forward(self.h_s)
            
            outputs.append(y_hat)
            decoder_caches.append((idx, attn_cache, dec_cache, ann_cache))
            
        caches = (encoder_caches, decoder_caches, encoder_outputs)
        return torch.cat(outputs, dim=0), caches

    def backward(self, predictions, targets, caches):
        encoder_caches, decoder_caches, encoder_outputs = caches
        seq_len = targets.shape[0]

        # Initialize gradients for passing backward through time
        dh_dec_next = torch.zeros(1, self.hidden_dim)
        dc_dec_next = torch.zeros(1, self.hidden_dim)
        
        # We accumulate gradients for the encoder across all decoder steps
        dEncoder_outputs = torch.zeros_like(encoder_outputs)

        # ----------------------------------------
        # BPTT: DECODER (Reverse Time Loop)
        # ----------------------------------------
        for t in reversed(range(len(decoder_caches))):
            idx, attn_cache, dec_cache, ann_cache = decoder_caches[t]
            
            # 1. Output Layer gradient: dZ = (y_hat - y) / seq_len
            dZ = (predictions[t] - targets[t]).unsqueeze(0) / seq_len
            
            # 2. ANN backward
            dh_ann = self.ANN.backward(dZ, ann_cache)
            
            # Combine gradients flowing into Decoder hidden state
            dh_dec_total = dh_ann + dh_dec_next
            
            # 3. Decoder LSTM backward
            dLstm_input, dc_dec_next, dh_dec_next = self.decoder.backward(dh_dec_total, dc_dec_next, dec_cache)
            
            # 4. Split input gradient (Embedding vs Context)
            embed_dim = self.tgt_embed.E.shape[1]
            dY_t = dLstm_input[:, :embed_dim]
            dContext = dLstm_input[:, embed_dim:]
            
            # 5. Target Embedding backward
            self.tgt_embed.backward(dY_t, idx)
            
            # 6. Attention backward
            dQuery, dKeys = self.attention.backward(dContext, attn_cache)
            
            # Accumulate gradient for next decoder step and encoder outputs
            dh_dec_next += dQuery
            dEncoder_outputs += dKeys

        # ----------------------------------------
        # BPTT: ENCODER (Reverse Time Loop)
        # ----------------------------------------
        dh_enc_next = torch.zeros(1, self.hidden_dim)
        dc_enc_next = torch.zeros(1, self.hidden_dim)
        
        for t in reversed(range(len(encoder_caches))):
            idx, enc_cache = encoder_caches[t]
            
            # Combine local gradient from Attention with gradient from next encoder step
            dh_enc_total = dEncoder_outputs[t].unsqueeze(0) + dh_enc_next
            
            # Encoder LSTM backward
            dX_t, dc_enc_next, dh_enc_next = self.encoder.backward(dh_enc_total, dc_enc_next, enc_cache)
            
            # Source Embedding backward
            self.src_embed.backward(dX_t, idx)
        
    def update_weights(self, lr):
        self.src_embed.update_weights(lr)
        self.tgt_embed.update_weights(lr)
        self.encoder.update_weights(lr)
        self.decoder.update_weights(lr)
        self.attention.update_weights(lr)
        self.ANN.update_weights(lr)

# ==========================================
# 4. TRAINING AND INFERENCE
# ==========================================

embed_dim = 16
hidden_dim = 32

model = Seq2seq(len(eng_vocab), len(urdu_vocab), embed_dim, hidden_dim)
criterion = CrossEntropyLoss()

epochs = 1000
learning_rate = 0.1

print("Starting Manual BPTT Training...\n")

for epoch in range(epochs):
    epoch_loss = 0
    
    for eng_sent, urdu_sent in raw_data:
        src_indices = [eng_vocab[w] for w in eng_sent.lower().split()]
        tgt_indices = [urdu_vocab["<SOS>"]] + [urdu_vocab[w] for w in urdu_sent.lower().split()] + [urdu_vocab["<EOS>"]]
        
        # 1. Forward Pass (now returns caches!)
        predictions, caches = model.forward(src_indices, tgt_indices)
        
        # 2. Compute targets and Scalar Loss
        target_words_to_predict = tgt_indices[1:]
        Y_one_hot = torch.zeros(len(target_words_to_predict), len(urdu_vocab))
        for i, word_idx in enumerate(target_words_to_predict):
            Y_one_hot[i, word_idx] = 1.0
            
        loss = criterion.forward(predictions, Y_one_hot)
        epoch_loss += loss.item()
        
        # 3. MANUAL BACKWARD PASS & UPDATE
        model.backward(predictions, Y_one_hot, caches)
        model.update_weights(lr=learning_rate)
        
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Average Loss: {(epoch_loss/len(raw_data)):.4f}")

# Inference Function
def translate(model, english_sentence, max_length=15):
    words = english_sentence.lower().split()
    src_indices = [eng_vocab[w] for w in words if w in eng_vocab]
    
    model.c_s = torch.zeros(1, model.hidden_dim)
    model.h_s = torch.zeros(1, model.hidden_dim)

    encoder_outputs = [] 
    for idx in src_indices:
        x_t = model.src_embed.forward(idx).unsqueeze(0) 
        model.c_s, model.h_s, _ = model.encoder.forward(x_t, model.c_s, model.h_s) # Ignore cache
        encoder_outputs.append(model.h_s) 
        
    encoder_outputs = torch.cat(encoder_outputs, dim=0) 

    predicted_words = []
    current_word_idx = urdu_vocab["<SOS>"]
    
    for t in range(max_length):
        y_t = model.tgt_embed.forward(current_word_idx).unsqueeze(0)
        
        context_vector, _ = model.attention.forward(model.h_s, encoder_outputs) # Ignore cache
        lstm_input = torch.cat([y_t, context_vector], dim=1)
        
        model.c_s, model.h_s, _ = model.decoder.forward(lstm_input, model.c_s, model.h_s) # Ignore cache
        
        probabilities, _ = model.ANN.forward(model.h_s) # Ignore cache
        best_guess_idx = torch.argmax(probabilities, dim=1).item()
        
        if best_guess_idx == urdu_vocab["<EOS>"]:
            break
            
        predicted_words.append(urdu_idx_to_word[best_guess_idx])
        current_word_idx = best_guess_idx

    return " ".join(predicted_words)

# Quick Test


Starting Manual BPTT Training...

Epoch 50/1000 | Average Loss: 2.5714
Epoch 100/1000 | Average Loss: 1.9403
Epoch 150/1000 | Average Loss: 1.3258
Epoch 200/1000 | Average Loss: 0.9056
Epoch 250/1000 | Average Loss: 0.6361
Epoch 300/1000 | Average Loss: 0.5070
Epoch 350/1000 | Average Loss: 0.4295
Epoch 400/1000 | Average Loss: 0.3655
Epoch 450/1000 | Average Loss: 0.3012
Epoch 500/1000 | Average Loss: 0.2254
Epoch 550/1000 | Average Loss: 0.1576
Epoch 600/1000 | Average Loss: 0.0892
Epoch 650/1000 | Average Loss: 0.0559
Epoch 700/1000 | Average Loss: 0.0401
Epoch 750/1000 | Average Loss: 0.0311
Epoch 800/1000 | Average Loss: 0.0254
Epoch 850/1000 | Average Loss: 0.0214
Epoch 900/1000 | Average Loss: 0.0185
Epoch 950/1000 | Average Loss: 0.0163
Epoch 1000/1000 | Average Loss: 0.0145


In [7]:
print("\nTesting Inference:")
print("Input: he plays hockey")
print("Output:", translate(model, "he plays hockey"))


Testing Inference:
Input: he plays hockey
Output: wo cricket khelta hai
